In [2]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from docx import Document

In [3]:
import faiss

candidate_embeddings = np.load(
    "../models/candidate_embeddings.npy"
)

candidate_embeddings.shape

(100000, 384)

In [4]:
faiss.normalize_L2(candidate_embeddings)

In [5]:
dimension = candidate_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(candidate_embeddings)

print("Number of vectors:", index.ntotal)

Number of vectors: 100000


In [6]:
faiss.write_index(
    index,
    "../models/faiss_index.bin"
)

In [8]:
# Instead of using the raw docx, we use a concentrated semantic query 
# targeting the exact profile the founders are looking for.

job_text = """
Senior AI Engineer with 5 to 9 years of product-company experience. 
Strong expertise in machine learning infrastructure, embeddings, retrieval systems, and ranking. 
Hands-on production experience with vector databases (Pinecone, FAISS, Weaviate, Qdrant) and distributed systems. 
Capable of building end-to-end evaluation frameworks for ML systems.
"""
print(job_text)


Senior AI Engineer with 5 to 9 years of product-company experience. 
Strong expertise in machine learning infrastructure, embeddings, retrieval systems, and ranking. 
Hands-on production experience with vector databases (Pinecone, FAISS, Weaviate, Qdrant) and distributed systems. 
Capable of building end-to-end evaluation frameworks for ML systems.



In [7]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
job_embedding = model.encode([job_text])

job_embedding.shape

(1, 384)

In [10]:
faiss.normalize_L2(job_embedding)

In [11]:
k = 500

distances, indices = index.search(
    job_embedding,
    k
)

In [12]:
print(indices.shape)
print(distances.shape)

(1, 500)
(1, 500)


In [13]:
top_indices = indices[0]

top_indices[:10]

array([58411, 51485, 58151, 89011, 63584, 85556, 42369, 33178, 84680,
       71032])

In [14]:
candidate_df = pd.read_csv(
    "../data/candidate_texts.csv"
)

In [15]:
top_candidates = candidate_df.iloc[top_indices]

top_candidates.head()

,candidate_id,candidate_text
58411,CAND_0058412,Current title: Senior Software Engineer (ML)\n...
51485,CAND_0051486,Current title: AI Specialist\nSummary: Data sc...
58151,CAND_0058152,Current title: Data Scientist\nSummary: Data s...
89011,CAND_0089012,Current title: AI Specialist\nSummary: Data sc...
63584,CAND_0063585,Current title: AI Research Engineer\nSummary: ...


In [16]:
top_candidates = top_candidates.copy()

top_candidates["similarity_score"] = distances[0]

In [17]:
top_candidates = top_candidates.sort_values(
    "similarity_score",
    ascending=False
)

In [18]:
top_candidates[
    ["candidate_id", "similarity_score"]
].head(20)

,candidate_id,similarity_score
58411,CAND_0058412,0.745611
51485,CAND_0051486,0.744759
58151,CAND_0058152,0.744204
89011,CAND_0089012,0.742055
63584,CAND_0063585,0.740394
85556,CAND_0085557,0.739397
42369,CAND_0042370,0.737439
33178,CAND_0033179,0.734471
84680,CAND_0084681,0.734335
71032,CAND_0071033,0.734322


In [19]:
top_candidates.to_csv(
    "../outputs/faiss_top500.csv",
    index=False
)